# Lead AI · Informatique pour l'IA

**Objectif :** GPU, Kubernetes, Helm et calcul distribué avec Ray.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## GPU : architecture et optimisation

Le GPU est la ressource la plus chère d'un projet d'IA : une carte H100 coûte autour de 30 000 €, ou 2 à 4 € de l'heure dans le cloud. Un Lead AI doit savoir ce qui la limite, pour choisir le bon matériel et ne pas payer des GPU qui attendent.

## Pourquoi un GPU

Un processeur (CPU) a quelques dizaines de cœurs puissants. Un GPU en a des **milliers** de plus simples, plus des **Tensor Cores** spécialisés dans le produit matriciel en basse précision. Or un réseau de neurones, c'est presque uniquement des produits matriciels.

| Carte | Calcul bf16 (dense) | Mémoire | Bande passante |
| --- | --- | --- | --- |
| NVIDIA T4 (Colab gratuit) | 65 TFLOPS | 16 Go | 320 Go/s |
| NVIDIA A100 | 312 TFLOPS | 80 Go | 2 000 Go/s |
| NVIDIA H100 SXM | 989 TFLOPS | 80 Go | 3 350 Go/s |

## Calcul ou mémoire : le modèle du toit

Chaque opération lit des données en mémoire, puis calcule. Son **intensité arithmétique** est le nombre d'opérations faites par octet lu. Le **modèle du toit** (*roofline*) en déduit la performance atteignable :

$$
\text{perf} = \min\big(\text{pic de calcul},\ \text{intensité} \times \text{bande passante}\big)
$$

Sous un certain seuil d'intensité (environ 300 opérations par octet sur H100), on est **limité par la mémoire** : le GPU attend les données. Au-dessus, on est **limité par le calcul**.

*Animation interactive « roofline » : à voir sur la plateforme web.*

- Un grand produit matriciel (entraînement, préremplissage d'un LLM) a une forte intensité : il utilise vraiment le GPU.
- Un produit matrice × vecteur (décodage d'un LLM, un token à la fois) a une intensité proche de 1 : il est limité par la mémoire. C'est pourquoi on regroupe les requêtes (vague 6) et pourquoi la quantification accélère l'inférence : moins d'octets à lire.

## Les leviers d'optimisation

- **Précision mixte** (bf16 ou fp16) : 2 fois moins de mémoire, et les Tensor Cores. En PyTorch : `torch.autocast("cuda", dtype=torch.bfloat16)`.
- **Taille de lot** : la plus grande qui tient en mémoire, pour augmenter l'intensité.
- **Fusion d'opérations** : `torch.compile` regroupe plusieurs petites opérations pour éviter des allers-retours en mémoire ; FlashAttention fait de même pour l'attention.
- **Ne pas affamer le GPU** : un chargement de données trop lent (`DataLoader` avec `num_workers` et `pin_memory`) laisse le GPU à 30 % d'utilisation.
- **Mesurer** : `nvidia-smi` pour l'utilisation et la mémoire, le profileur PyTorch pour savoir où passe le temps.

## Estimer un entraînement

Entraîner un Transformer coûte environ **6 opérations par paramètre et par token** (2 pour la passe avant, 4 pour la rétropropagation). Le taux d'utilisation réel, la **MFU** (*model FLOPs utilization*), dépasse rarement 40 à 50 %.

$$
\text{durée} \approx \frac{6\, N\, D}{n_{\text{GPU}} \times \text{pic} \times \text{MFU}}
$$

> **À retenir** : le GPU est limité soit par le calcul, soit par la mémoire. Entraînement et préremplissage : calcul. Décodage : mémoire. Précision mixte, grands lots, fusion d'opérations et chargement de données efficace. Coût d'un entraînement ≈ 6 N D opérations.

**À faire :**

- [ ] Sur Google Colab avec un GPU T4, lancer nvidia-smi et repérer la mémoire et l'utilisation
- [ ] Mesurer le temps d'un produit de matrices 4096 × 4096 avec PyTorch en float32 puis en bfloat16 ou float16
- [ ] Entraîner le CNN de la vague 4 avec et sans torch.autocast, et comparer la durée d'une époque

### Exercice 1 · Le modèle du toit

Écris `intensite_matmul(m, n, k, octets=2)` : l'intensité arithmétique du produit d'une matrice (m, k) par une matrice (k, n), qui fait 2·m·n·k opérations et lit ou écrit (m·k + k·n + m·n) nombres de `octets` octets. Puis `regime(intensite, pic_tflops, bande_go_s)` qui renvoie `"calcul"` si l'intensité atteint le seuil pic / bande passante (attention aux unités : téra = 1e12, giga = 1e9), sinon `"mémoire"`. Enfin `perf_tflops(intensite, pic_tflops, bande_go_s)`, la performance atteignable en TFLOPS selon le modèle du toit.

<details><summary>Indice</summary>

Le seuil vaut pic_tflops × 1e12 / (bande_go_s × 1e9) opérations par octet. perf = min(pic, intensité × bande) en convertissant la bande en TFLOPS.
</details>

In [ ]:
def intensite_matmul(m, n, k, octets=2):
    pass

def regime(intensite, pic_tflops, bande_go_s):
    pass

def perf_tflops(intensite, pic_tflops, bande_go_s):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
I = intensite_matmul(4096, 4096, 4096)
assert abs(I - 4096 / 3) < 1e-9, f"Pour 4096³ en 16 bits, l'intensité vaut 2·4096³ / (2·3·4096²) = 1365,3 ; tu as {I}."
assert abs(intensite_matmul(10, 1, 10, octets=4) - 200 / (4 * 120)) < 1e-12, "Vérifie la formule : 2mnk / (octets × (mk + kn + mn))."
assert regime(I, 989, 3350) == "calcul" and regime(intensite_matmul(4096, 1, 4096), 989, 3350) == "mémoire", "Le grand produit matriciel est limité par le calcul, le produit matrice × vecteur par la mémoire."
assert regime(989e12 / 3350e9, 989, 3350) == "calcul", "Au seuil exact, on considère qu'on atteint le calcul (>=)."
assert perf_tflops(I, 989, 3350) == 989, "Au-dessus du seuil, la performance plafonne au pic."
assert abs(perf_tflops(1.0, 989, 3350) - 3.35) < 1e-9, "Avec une intensité de 1, on atteint 1 × 3 350 Go/s = 3,35 TFLOPS."
print("Bravo, tout est correct.")

### Exercice 2 · Combien de temps pour entraîner un LLM ?

Écris `duree_jours(n_params, n_tokens, n_gpus, pic_tflops, mfu)` avec la règle des 6·N·D opérations. Puis calcule pour Llama 3 8B (8e9 paramètres, 15e12 tokens) sur 1 024 H100 (989 TFLOPS) à une MFU de 0,4 : `jours`, et `cout_euros`, le coût à 2,50 € par GPU et par heure.

<details><summary>Indice</summary>

6 * n_params * n_tokens opérations ; le débit effectif vaut n_gpus × pic_tflops × 1e12 × mfu opérations par seconde.
</details>

In [ ]:
def duree_jours(n_params, n_tokens, n_gpus, pic_tflops, mfu):
    pass

jours = None
cout_euros = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
attendu = 6 * 8e9 * 15e12 / (1024 * 989e12 * 0.4) / 86400
assert abs(duree_jours(8e9, 15e12, 1024, 989, 0.4) - attendu) < 1e-9, f"duree_jours doit valoir {attendu:.2f} jours pour Llama 3 8B."
assert abs(duree_jours(1e9, 1e9, 1, 1, 1) - 6e18 / 1e12 / 86400) < 1e-9, "6 × N × D opérations divisées par (GPUs × pic × 1e12 × MFU), puis par 86 400."
assert abs(jours - attendu) < 1e-9, "jours doit utiliser les valeurs de Llama 3 8B."
assert abs(cout_euros - attendu * 24 * 1024 * 2.5) < 1e-3, "cout_euros = jours × 24 h × 1 024 GPU × 2,50 €."
print("Bravo, tout est correct.")

---
## Kubernetes et Helm

Docker fait tourner un conteneur sur une machine. En production, on veut des dizaines de conteneurs sur des dizaines de machines, qui redémarrent quand ils plantent, se multiplient quand la charge monte, et se mettent à jour sans coupure. C'est le rôle d'un **orchestrateur**, et **Kubernetes** (K8s) est le standard. Les services gérés s'appellent EKS (AWS), GKE (Google) et AKS (Azure).

## Les objets à connaître

| Objet | Rôle |
| --- | --- |
| **Pod** | Un ou plusieurs conteneurs lancés ensemble ; l'unité de base |
| **Deployment** | Maintient N copies identiques d'un pod et gère les mises à jour progressives |
| **Service** | Une adresse stable qui répartit le trafic entre les pods |
| **Ingress** | L'entrée HTTP depuis l'extérieur (domaine, HTTPS, chemins) |
| **ConfigMap** et **Secret** | La configuration et les secrets, injectés en variables d'environnement |
| **HorizontalPodAutoscaler** | Ajoute ou retire des pods selon la charge |
| **Job** et **CronJob** | Une tâche ponctuelle ou planifiée (un entraînement, un scoring de nuit) |

## Un Deployment pour l'API de fraude

```python
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fraude-api
spec:
  replicas: 3
  selector:
    matchLabels: {app: fraude-api}
  template:
    metadata:
      labels: {app: fraude-api}
    spec:
      containers:
        - name: fraude-api
          image: 123456789.dkr.ecr.eu-west-3.amazonaws.com/fraude-api:1.2.0
          ports: [{containerPort: 8000}]
          resources:
            requests: {cpu: "500m", memory: "512Mi"}      # ce qui est réservé
            limits:   {cpu: "1",    memory: "1Gi"}        # le maximum autorisé
          readinessProbe:                                  # prêt à recevoir du trafic ?
            httpGet: {path: /sante, port: 8000}
```

On applique ce fichier avec `kubectl apply -f deployment.yaml`. Kubernetes compare en permanence l'**état désiré** (3 copies) à l'**état réel**, et corrige l'écart. C'est le principe central : on décrit le résultat voulu, pas les étapes.

## La montée en charge automatique

*Animation interactive « hpa » : à voir sur la plateforme web.*

L'**HorizontalPodAutoscaler** calcule le nombre de pods nécessaire à partir d'une métrique (CPU, requêtes par seconde, longueur d'une file) :

$$
\text{replicas} = \left\lceil \text{replicas actuels} \times \frac{\text{valeur mesurée}}{\text{valeur cible}} \right\rceil
$$

borné par un minimum et un maximum, avec une tolérance de 10 % pour éviter les oscillations.

## Les GPU dans Kubernetes

Un pod demande un GPU avec `limits: {nvidia.com/gpu: 1}` ; le plugin NVIDIA place le pod sur un nœud qui en a un. Les GPU sont chers : on les met dans un groupe de nœuds séparé, qui monte à zéro quand rien ne tourne.

## Helm : des modèles de manifestes

Écrire des dizaines de fichiers YAML presque identiques (développement, recette, production) est fastidieux. **Helm** est le gestionnaire de paquets de Kubernetes : un **chart** contient des gabarits avec des trous, et un fichier `values.yaml` par environnement les remplit.

```python
# templates/deployment.yaml (extrait)
spec:
  replicas: {{ .Values.replicas }}
  containers:
    - image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"

helm install fraude ./chart -f values-prod.yaml
helm upgrade fraude ./chart --set image.tag=1.3.0     # mise à jour progressive
helm rollback fraude 1                                 # retour arrière
```

> **À retenir** : on décrit l'état désiré (Deployment, Service, Ingress), Kubernetes le maintient. Requests et limits pour les ressources, readinessProbe pour le trafic, HPA pour la charge. Helm pour réutiliser les manifestes d'un environnement à l'autre.

**À faire :**

- [ ] Installer kind ou minikube et créer un cluster local
- [ ] Déployer l'image de l'API de la vague 6 avec un Deployment et un Service, puis l'appeler avec kubectl port-forward
- [ ] Ajouter un HorizontalPodAutoscaler et générer de la charge pour voir les pods se multiplier
- [ ] Transformer les manifestes en chart Helm avec un values.yaml par environnement

### Exercice 1 · Générer un Deployment

Écris `manifeste(nom, image, replicas, cpu, memoire, gpu=0, port=8000)` qui renvoie le dictionnaire d'un Deployment : `apiVersion` `apps/v1`, `kind` `Deployment`, `metadata.name`, `spec.replicas`, `spec.selector.matchLabels` = `{"app": nom}`, et `spec.template` avec `metadata.labels` = `{"app": nom}` et `spec.containers`, une liste d'un conteneur : `name`, `image`, `ports` = `[{"containerPort": port}]`, `resources` avec `requests` et `limits` égaux à `{"cpu": cpu, "memory": memoire}` (plus `"nvidia.com/gpu": gpu` dans `limits` si gpu > 0), et `readinessProbe` = `{"httpGet": {"path": "/sante", "port": port}}`.

<details><summary>Indice</summary>

Construis le dictionnaire limits à part, ajoute la clé GPU si besoin, puis assemble la structure imbriquée en suivant l'exemple YAML de la leçon.
</details>

In [ ]:
def manifeste(nom, image, replicas, cpu, memoire, gpu=0, port=8000):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
m = manifeste("fraude-api", "reg/fraude-api:1.2.0", 3, "500m", "512Mi")
assert m["apiVersion"] == "apps/v1" and m["kind"] == "Deployment" and m["metadata"] == {"name": "fraude-api"}, "Vérifie apiVersion, kind et metadata.name."
assert m["spec"]["replicas"] == 3 and m["spec"]["selector"] == {"matchLabels": {"app": "fraude-api"}}, "spec.replicas et spec.selector.matchLabels."
t = m["spec"]["template"]
assert t["metadata"]["labels"] == {"app": "fraude-api"}, "Les labels du gabarit doivent correspondre au sélecteur, sinon le Deployment ne trouve pas ses pods."
c = t["spec"]["containers"]
assert isinstance(c, list) and len(c) == 1, "containers est une liste d'un conteneur."
c = c[0]
assert c["name"] == "fraude-api" and c["image"] == "reg/fraude-api:1.2.0" and c["ports"] == [{"containerPort": 8000}], "Vérifie name, image et ports."
assert c["resources"] == {"requests": {"cpu": "500m", "memory": "512Mi"}, "limits": {"cpu": "500m", "memory": "512Mi"}}, "requests et limits avec cpu et memory ; pas de GPU si gpu = 0."
assert c["readinessProbe"] == {"httpGet": {"path": "/sante", "port": 8000}}, "readinessProbe sur /sante."
g = manifeste("llm", "vllm:0.6", 1, "8", "64Gi", gpu=2, port=9000)["spec"]["template"]["spec"]["containers"][0]
assert g["resources"]["limits"]["nvidia.com/gpu"] == 2 and "nvidia.com/gpu" not in g["resources"]["requests"], "Avec gpu > 0, ajoute nvidia.com/gpu dans limits seulement."
assert g["ports"] == [{"containerPort": 9000}] and g["readinessProbe"]["httpGet"]["port"] == 9000, "Le port doit être utilisé partout."
print("Bravo, tout est correct.")

### Exercice 2 · L'autoscaler

Écris `replicas_cibles(actuels, mesure, cible, min_r=1, max_r=10, tolerance=0.1)` : si le rapport mesure / cible est à moins de `tolerance` de 1 (inclus), garde `actuels` ; sinon, calcule ⌈actuels × mesure / cible⌉ (avec `math.ceil`). Dans tous les cas, borne le résultat entre `min_r` et `max_r`.

<details><summary>Indice</summary>

rapport = mesure / cible ; si abs(rapport − 1) <= tolerance on garde actuels ; puis max(min_r, min(max_r, …)).
</details>

In [ ]:
import math

def replicas_cibles(actuels, mesure, cible, min_r=1, max_r=10, tolerance=0.1):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert replicas_cibles(4, 90, 60) == 6, "4 pods à 90 % pour une cible de 60 % : ⌈4 × 1,5⌉ = 6."
assert replicas_cibles(4, 62, 60) == 4, "62 / 60 est dans la tolérance de 10 % : on ne bouge pas."
assert replicas_cibles(4, 65, 60) == 4, "65 / 60 ≈ 1,08 : dans la tolérance, on ne bouge pas."
assert replicas_cibles(4, 70, 60) == 5, "70 / 60 ≈ 1,17 : hors tolérance, ⌈4 × 1,17⌉ = 5."
assert replicas_cibles(4, 30, 60) == 2, "À 30 % pour 60 %, on divise par deux : 2 pods."
assert replicas_cibles(4, 250, 60) == 10, "Le résultat est plafonné à max_r."
assert replicas_cibles(2, 5, 60, min_r=2) == 2, "Le résultat ne descend pas sous min_r."
assert replicas_cibles(3, 61, 60, tolerance=0) == 4, "Avec une tolérance nulle, ⌈3 × 61/60⌉ = 4."
print("Bravo, tout est correct.")

### Exercice 3 · Un gabarit à la Helm

Écris `rendre(gabarit, valeurs)` qui remplace chaque `{{ .Values.chemin.vers.cle }}` (espaces facultatifs autour) par la valeur correspondante dans le dictionnaire imbriqué `valeurs`, convertie en texte. Si une clé manque, lève une `KeyError` dont le message contient le chemin complet (par exemple `image.tag`). Utilise `re.sub` avec une fonction.

<details><summary>Indice</summary>

re.sub(r"\{\{\s*\.Values\.([\w.]+)\s*\}\}", fonction, gabarit) : la fonction reçoit l'objet match et suit le chemin m.group(1).split(".").
</details>

In [ ]:
import re

def rendre(gabarit, valeurs):
    pass

gabarit = """spec:
  replicas: {{ .Values.replicas }}
  containers:
    - image: "{{ .Values.image.repository }}:{{.Values.image.tag}}"
"""
valeurs = {"replicas": 3, "image": {"repository": "reg/fraude-api", "tag": "1.2.0"}}

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = rendre(gabarit, valeurs)
assert r is not None and "replicas: 3" in r and '"reg/fraude-api:1.2.0"' in r and "{{" not in r, f"Toutes les valeurs doivent être remplacées ; tu obtiens :\n{r}"
assert rendre("a={{ .Values.x }} b={{.Values.x}}", {"x": True}) == "a=True b=True", "Les espaces autour sont facultatifs, et la valeur est convertie en texte."
try:
    rendre("{{ .Values.image.digest }}", valeurs); ok = False
except KeyError as e:
    ok = "image.digest" in str(e)
assert ok, "Une clé manquante doit lever une KeyError qui nomme le chemin complet."
assert rendre("rien à remplacer", {}) == "rien à remplacer", "Un texte sans gabarit reste inchangé."
print("Bravo, tout est correct.")

---
## Calcul distribué : Ray et deep learning distribué

Quand un calcul ne tient plus sur une machine (trop de données, trop d'hyperparamètres à essayer, un modèle trop gros pour un GPU), il faut le **distribuer**. Deux outils dominent : **Ray** pour distribuer du Python en général, et les techniques de **parallélisme** de PyTorch pour l'entraînement des gros modèles.

## Ray : du Python sur un cluster

Ray transforme une fonction en **tâche** distante et une classe en **acteur** (un processus qui garde un état), avec un seul décorateur. Le même code tourne sur un portable ou sur 100 machines.

```python
import ray
ray.init()                                    # local, ou ray.init(address="ray://cluster:10001")

@ray.remote
def scorer(lot):
    return modele.predict(lot)

futurs = [scorer.remote(lot) for lot in lots]  # lancés en parallèle, partout dans le cluster
resultats = ray.get(futurs)
```

Au-dessus, des bibliothèques prêtes à l'emploi : **Ray Tune** (recherche d'hyperparamètres distribuée), **Ray Train** (entraînement distribué), **Ray Data** (traitement de gros volumes), **Ray Serve** (servir des modèles). OpenAI et Uber l'utilisent à très grande échelle.

## Le parallélisme de données

La forme la plus courante pour entraîner : chaque GPU a une **copie complète** du modèle et traite une **part du lot**. Après la rétropropagation, les GPU font la **moyenne de leurs gradients** (une opération collective appelée *all-reduce*), et appliquent tous la même mise à jour. Mathématiquement, c'est exactement la même descente de gradient que sur un seul GPU avec le lot complet. En PyTorch : `DistributedDataParallel` (DDP), lancé avec `torchrun --nproc_per_node=8 train.py`.

## Quand le modèle ne tient plus sur un GPU

Rappel (vague 5) : entraîner avec Adam en précision mixte demande environ **16 octets par paramètre** : 2 pour les poids en 16 bits, 2 pour les gradients, 12 pour les états de l'optimiseur (copie des poids en 32 bits et deux moments). Un modèle de 7 milliards de paramètres demande donc 112 Go, plus qu'un H100.

**ZeRO** (DeepSpeed) et **FSDP** (PyTorch) découpent ces données entre les N GPU au lieu de les dupliquer :

| Étape | Ce qui est découpé | Mémoire par GPU |
| --- | --- | --- |
| Sans ZeRO | rien | 16 Ψ |
| ZeRO-1 | états de l'optimiseur | 4 Ψ + 12 Ψ / N |
| ZeRO-2 | + gradients | 2 Ψ + 14 Ψ / N |
| ZeRO-3 (FSDP) | + poids | 16 Ψ / N |

(Ψ = nombre de paramètres, en octets par paramètre.)

*Animation interactive « zero » : à voir sur la plateforme web.*

Pour les plus gros modèles, on combine avec le **parallélisme de tenseur** (chaque matrice est coupée entre GPU d'une même machine) et le **parallélisme de pipeline** (les couches sont réparties en étages). C'est ce qu'on appelle le parallélisme 3D.

> **À retenir** : Ray pour distribuer du Python (tâches, acteurs, Tune, Serve). Parallélisme de données = copies du modèle + moyenne des gradients, mathématiquement identique. ZeRO et FSDP découpent poids, gradients et états de l'optimiseur pour faire tenir les gros modèles.

**À faire :**

- [ ] Installer ray, lancer ray.init() et paralléliser un scoring du jeu Immo avec @ray.remote
- [ ] Faire une recherche d'hyperparamètres avec Ray Tune sur 4 cœurs et comparer la durée avec GridSearchCV
- [ ] Lire le tutoriel « Getting Started with Distributed Data Parallel » de PyTorch et lancer l'exemple avec torchrun sur CPU

### Exercice 1 · Le parallélisme de données, vérifié

Pour une régression linéaire, le gradient de l'erreur quadratique moyenne vaut 2/n · Xᵀ(Xw − y) (fonction `gradient` fournie). Écris `gradient_parallele(X, y, w, k)` : découpe les données en `k` parts avec `np.array_split` (une par « GPU »), calcule le gradient de chaque part, puis combine-les en une **moyenne pondérée par la taille des parts** (l'all-reduce). Vérifie que le résultat est identique au gradient complet.

<details><summary>Indice</summary>

np.array_split(X, k) et np.array_split(y, k) ; np.average(grads, axis=0, weights=tailles).
</details>

In [ ]:
import numpy as np

def gradient(X, y, w):
    return 2 / len(X) * X.T @ (X @ w - y)

rng = np.random.default_rng(0)
X = rng.normal(size=(1003, 5))
y = X @ np.array([1.0, -2.0, 0.5, 3.0, 0.0]) + rng.normal(0, 0.1, 1003)
w = np.zeros(5)

def gradient_parallele(X, y, w, k):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
for k in [1, 2, 7, 8]:
    g = gradient_parallele(X, y, w * 0 + 0.3, k)
    assert g is not None and np.allclose(g, gradient(X, y, w * 0 + 0.3)), f"Avec k = {k}, le gradient parallèle doit être identique au gradient complet. Pondère chaque part par sa taille (les parts n'ont pas toutes la même taille : 1003 n'est pas divisible par 7 ou 8)."
print("Bravo, tout est correct.")

### Exercice 2 · La mémoire avec ZeRO

Écris `memoire_zero_go(n_params, n_gpus, etape)` qui renvoie la mémoire par GPU en Go (1 Go = 1e9 octets) selon le tableau de la leçon, pour `etape` valant 0, 1, 2 ou 3. Puis calcule `gpus_min_70b`, le plus petit nombre de GPU de 80 Go qui permet d'entraîner un modèle de 70e9 paramètres avec ZeRO-3, sans compter les activations.

<details><summary>Indice</summary>

Un dictionnaire des quatre formules, indexé par etape. Pour le minimum : essaie n = 1, 2, 3… jusqu'à tenir dans 80 Go.
</details>

In [ ]:
def memoire_zero_go(n_params, n_gpus, etape):
    pass

gpus_min_70b = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert memoire_zero_go(7e9, 8, 0) == 112, "Sans ZeRO : 16 octets par paramètre, soit 112 Go pour 7B."
assert abs(memoire_zero_go(7e9, 8, 1) - (28 + 84 / 8)) < 1e-9, "ZeRO-1 : 4Ψ + 12Ψ / N."
assert abs(memoire_zero_go(7e9, 8, 2) - (14 + 98 / 8)) < 1e-9, "ZeRO-2 : 2Ψ + 14Ψ / N."
assert abs(memoire_zero_go(7e9, 8, 3) - 14) < 1e-9, "ZeRO-3 : 16Ψ / N, soit 14 Go pour 7B sur 8 GPU."
assert gpus_min_70b == 14, f"16 × 70e9 / N ≤ 80e9 donne N ≥ 14 ; tu as {gpus_min_70b}."
print("Bravo, tout est correct.")